In [0]:
%run /Workspace/consolidated_pipeline/02_dimension_data_processing/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','customers','Data Source')

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")
print(catalog,data_source)

In [0]:
base_url=f"/Volumes/{catalog}/{bronze_schema}/sports_volume/{data_source}.csv"

In [0]:
from pyspark.sql.functions import current_timestamp
df=spark.read.format('csv').option('header',True).option('inferschema',True).load(base_url).withColumn('read_stamp_time',current_timestamp())
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write.format('delta').mode('overwrite').option('delta.enableChangeDataFeed',"true"). \
saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

silver processing

In [0]:
df_bronze=spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source};')


In [0]:
from pyspark.sql.functions import col
df_duplicates=df_bronze.groupBy('customer_id').count().filter(col('count')>1)
display(df_duplicates)

In [0]:
df_silver=df_bronze.dropDuplicates(['customer_id'])

In [0]:
from pyspark.sql.functions import trim
display(df_silver.filter(col("customer_name")!= trim(col("customer_name"))))

In [0]:
df_silver=df_silver.withColumn('customer_name',trim(col('customer_name')))
display(df_silver)

In [0]:
display(df_silver.filter(col("customer_name")!= trim(col("customer_name"))))

In [0]:
df_silver.select('city').distinct().show()

In [0]:
from pyspark.sql.functions import col, when

city_mapping = {
    'Bengaluruu':'Bengaluru',
    'Bengalore':'Bengaluru',
    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',
    'NewDelhee':'New Delhi',
    'NewDelhi':'New Delhi',
    'NewDheli':'New Delhi'
}

allowed = ['Bengaluru','Hyderabad','New Delhi']

df_silver = df_silver.replace(city_mapping, subset=['city']) \
    .withColumn(
        'city',
        when(col('city').isin(allowed), col('city'))
        .otherwise(None)
    )

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
from pyspark.sql.functions import initcap
df_silver=df_silver.withColumn('customer_name', initcap(col('customer_name')))

In [0]:
names=df_silver.select('customer_name').distinct()

In [0]:
from pyspark.sql.functions import col
df_names=df_silver.select('customer_name').filter(col('city').isNull())
display(df_names)


In [0]:
df_silver.join(df_names,on='customer_name',how='inner')

In [0]:
df_cities=df_silver.select('city').distinct().filter(col('city').isNotNull())
display (df_cities)

In [0]:
unique_customers=df_names.distinct()
display(unique_customers)

In [0]:
missing_city=df_silver.filter(col("city").isNull())

In [0]:
df_base=df_silver

In [0]:
missing_city.join(df_silver,on='customer_name',how='left').orderBy(col('customer_name')).show()

In [0]:
df_silver_fix= {
    789521:'Hyderabad',
    789603:'New Delhi',
    789403:'New Delhi',
    789420:'Bengaluru'
}

In [0]:
df_fix=spark.createDataFrame([(k,v) for k,v in df_silver_fix.items()],['customer_id','fix_cities'])
display(df_fix)

In [0]:
from pyspark.sql.functions import coalesce
df_base=df_silver.join(df_fix,on='customer_id',how='left').withColumn('city',coalesce('city','fix_cities')).drop('fix_cities')


In [0]:
display(df_base)

In [0]:
from pyspark.sql.functions import concat_ws,lit
df_silver=df_base.withColumn('customer_id', col("customer_id").cast('string')).withColumn('customer',concat_ws('-',col('customer_name'),coalesce(col('city'),lit('Unknown')))).\
    withColumn('market',lit('India')).\
        withColumn('platform',lit('Sports Bar')).\
            withColumn('channel',lit('Acquisitions'))

In [0]:
display(df_silver)

In [0]:
df_silver.write.format('delta').mode('overwrite').option('delta.enableChangeDataFeed','true').option('overwriteSchema','true').saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

Gold Processing

In [0]:
df_silvers=spark.sql(f'select * from {catalog}.{silver_schema}.{data_source};')
display(df_silvers)

In [0]:
df_gold=df_silvers.select('customer_id','customer_name','city','customer','market','platform','channel')

In [0]:
display(df_gold)

In [0]:
df_gold.write.format('delta').mode('overwrite').option("overwriteSchema", "true").\
option('delta.enableChangeDataFeed','true').saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forName(spark,'fmcg.gold.dim_customers')
df_child_customers=spark.table('fmcg.gold.sb_dim_customers').select((col('customer_id').alias('customer_code')),
'customer',
'market',
'platform',
'channel')

upsert operations

In [0]:
delta_table.alias('target').merge(
    source=df_child_customers.alias('source'),
    condition='target.customer_code = source.customer_code'
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()